# Jailbreak classifier — updated issue #5
**One self-contained notebook; 10 Python cells.** Input CSVs are unchanged; the only exported file is `submission.csv` (`Id,TARGET`). Prompts are untrusted data, never executable instructions. TARGET >= 0.5 means jailbreak, otherwise benign.

## Reproduce
Python 3.11. Install `numpy==2.3.4 pandas==2.3.3 scipy==1.16.3 scikit-learn==1.7.2 nbformat==5.10.4 nbclient==0.10.2 ipykernel==7.0.1 threadpoolctl==3.6.0` in a virtual environment, select that environment's Jupyter kernel and Run All from the repository root. Dependencies are not auto-installed or downloaded by the notebook. CPU-only; no external datasets or model downloads.

## Research and development history
Before changes, web search covered Wang & Manning's text classification baselines and web fetch covered [updated issue #5](https://github.com/johndesantis/modeltrain/issues/5). The issue was checked again with GitHub API to obtain the updated file restrictions. [Original research](https://aclanthology.org/P12-2018/) and [scikit-learn text pipelines](https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_text_feature_extraction.html) informed the comparison.

Your reported public score is **0.95397**, with a target of **0.97**. No public score is claimed for this revision. The old four-member model scored 0.95761 on these local grouped folds. Development experiments (same folds, uncalibrated margins): RBF gamma 0.3/C10: 0.96012; gamma1/C10: 0.95556; gamma3/C10: 0.88882. A 3x3 grid gamma={0.03,0.1,0.3}, C={1,10,100} ranged 0.90977–0.96012. Polynomial `(1+cosine)^2` with C10 and C100 scored 0.96026. Selected C10 polynomial for this revision. These are **reused development folds**, not an untouched final holdout; selection can overestimate generalization. Safeguards are not evidence of improved public performance.

The final model uses word/character TF-IDF, a degree-two polynomial cosine kernel, and sigmoid calibration on **group-disjoint inner folds with fold-local vocabularies**. Calibration excludes each inner validation row from vocabulary/IDF/model fitting; outer evaluation remains separate. Likelihoods are not independently calibrated security guarantees. Lexical grouping cannot guarantee source/semantic independence.


In [1]:
from pathlib import Path
import csv, hashlib, json, platform, re, time, unicodedata, warnings
import numpy as np
import pandas as pd
import scipy, sklearn
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, brier_score_loss
from sklearn.neighbors import NearestNeighbors
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
warnings.filterwarnings('error', category=ConvergenceWarning)
SEED = 42
FOLDS = 5
NEAR_DUPLICATE_COSINE = 0.80
ROOT = Path.cwd()
START = time.monotonic()
# Avoid oversubscribed BLAS on CPU environments.
thread_limit = threadpool_limits(limits=2)
print('Python', platform.python_version(), '| sklearn', sklearn.__version__)
from io import StringIO
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import Normalizer
from sklearn.svm import SVC
INPUT_HASHES = {f: hashlib.sha256((ROOT/f).read_bytes()).hexdigest()
                for f in ['train.csv','test.csv','sample_submission.csv']}


Python 3.11.2 | sklearn 1.7.2


## 1. Strict CSV validation (IDs are never predictive features)

In [2]:
def require(condition, message):
    if not condition:
        raise ValueError(message)

def normalize(text):
    text = unicodedata.normalize('NFKC', text).casefold()
    text = ''.join(c for c in text if unicodedata.category(c) != 'Cf')
    return ' '.join(text.split())

def validate_frame(frame, columns):
    require(list(frame.columns) == columns, f'Expected columns {columns}')
    require(len(frame) > 0, 'Empty dataset')
    require(not frame.isna().any().any(), 'Missing values')
    require(frame['Id'].str.strip().ne('').all(), 'Blank IDs')
    require(frame['Id'].is_unique, 'Duplicate IDs')
    require(frame['Id'].eq(frame['Id'].str.strip()).all(), 'Whitespace in IDs')
    require(not frame['Id'].str.contains(r'[\x00-\x1f\x7f]', regex=True).any(), 'Control character in ID')
    if 'text' in columns:
        require(frame.text.map(normalize).ne('').all(), 'Empty normalized prompts')
        require(not frame.text.str.contains('\x00', regex=False).any(), 'NUL in prompt')
    if 'label' in columns:
        require(set(frame.label) == {'benign', 'jailbreak'}, 'Expected both binary labels')
    return frame

def parse_csv(stream, columns):
    reader = csv.reader(stream, strict=True)
    try:
        require(next(reader, None) == columns, 'Invalid CSV header')
        rows = list(reader)
    except csv.Error as exc:
        raise ValueError('Malformed CSV quoting') from exc
    require(all(len(row) == len(columns) for row in rows), 'Malformed CSV row width')
    return validate_frame(pd.DataFrame(rows, columns=columns, dtype=str), columns)

def load_csv(name, columns):
    with (ROOT / name).open(newline='', encoding='utf-8') as stream:
        return parse_csv(stream, columns)

def validate_relations(train, test, sample):
    require(set(train.Id).isdisjoint(test.Id), 'Train/test ID overlap')
    require(sample.Id.tolist() == test.Id.tolist(), 'Sample/test ID order mismatch')
    require(set(train.text.map(normalize)).isdisjoint(test.text.map(normalize)),
            'Normalized train/test text overlap')
    require(train.assign(key=train.text.map(normalize)).groupby('key').label.nunique().max() == 1,
            'Conflicting labels for identical normalized prompts')
    sample_scores = pd.to_numeric(sample.TARGET, errors='coerce').to_numpy()
    require(np.isfinite(sample_scores).all() and ((sample_scores >= 0) & (sample_scores <= 1)).all(),
            'Invalid sample targets')

train = load_csv('train.csv', ['Id', 'text', 'label'])
test = load_csv('test.csv', ['Id', 'text'])
sample = load_csv('sample_submission.csv', ['Id', 'TARGET'])
validate_relations(train, test, sample)
train['normalized'] = train.text.map(normalize)
test['normalized'] = test.text.map(normalize)
require(set(train.normalized).isdisjoint(test.normalized), 'Normalized train/test text overlap')
conflicts = train.groupby('normalized').label.nunique()
require(conflicts.max() == 1, 'Conflicting labels for identical normalized prompts')
duplicate_rows = int(train.normalized.duplicated().sum())
train = train.drop_duplicates('normalized').reset_index(drop=True)
texts = train.normalized.to_numpy()
y = train.label.map({'benign': 0, 'jailbreak': 1}).to_numpy()
print('Training:', len(train), '| test:', len(test), '| removed duplicates:', duplicate_rows)
print(train.label.value_counts().to_dict())

Training: 4000 | test: 1000 | removed duplicates: 0
{'benign': 2401, 'jailbreak': 1599}


## 2. Conservative near-duplicate grouping
Unsupervised hashing is used **only** for split grouping, never as predictive features. Connected components keep transitive template families in one fold. This is a lexical safeguard, not a semantic/source-independence guarantee.

In [3]:
from sklearn.feature_extraction.text import HashingVectorizer
# No learned vocabulary or IDF, no test prompts and no labels used for grouping.
group_vectorizer = HashingVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                                    n_features=2**18, alternate_sign=False, norm='l2')
group_matrix = group_vectorizer.transform(texts)
neighbor_index = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=2).fit(group_matrix)
graph = neighbor_index.radius_neighbors_graph(group_matrix,
    radius=1 - NEAR_DUPLICATE_COSINE + 1e-12, mode='connectivity')
_, groups = connected_components(graph, directed=False)
splits = list(StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
              .split(texts, y, groups))
for tr, va in splits:
    require(set(groups[tr]).isdisjoint(groups[va]), 'Group leakage')
    require(len(np.unique(y[tr])) == len(np.unique(y[va])) == 2, 'Single-class fold')
print('Groups:', len(np.unique(groups)), '| largest:', int(np.bincount(groups).max()))
print('Validation fold sizes:', [len(va) for _, va in splits])
# Audit only; test similarities never determine model weights or predictions.
nearest_distance, _ = neighbor_index.kneighbors(
    group_vectorizer.transform(test.normalized), n_neighbors=1)
near_test_count = int((nearest_distance[:, 0] <= 1 - NEAR_DUPLICATE_COSINE).sum())
print('Test prompts near a training prompt (audit only):', near_test_count)


Groups: 3855 | largest: 85
Validation fold sizes: [729, 803, 793, 791, 884]


Test prompts near a training prompt (audit only): 43


## 3. Polynomial-kernel model and leakage-safe calibration
Three inner grouped folds calibrate the model without exposing their validation text to feature fitting. The final refit uses all outer-training rows. Equal word/character L2 weighting; IDs and sample TARGET values are never features. Precomputed cosine products avoid the very slow sparse built-in RBF implementation. All functions/classes live in this notebook.

In [4]:
def vectorizer(view):
    common = dict(sublinear_tf=True, min_df=2, max_df=1.0, dtype=np.float64)
    if view == 'word':
        return TfidfVectorizer(ngram_range=(1, 2), token_pattern=r'(?u)\b\w+\b',
                               max_features=120000, **common)
    return TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 6),
                           max_features=180000, **common)

def polynomial_cosine(X, Y):
    return (1.0 + (X @ Y.T).toarray()) ** 2

def make_pipeline():
    return Pipeline([
        ('features', FeatureUnion([('word', vectorizer('word')), ('char', vectorizer('char'))])),
        ('normalize', Normalizer()),
        ('svm', SVC(C=10, kernel=polynomial_cosine, cache_size=512, random_state=SEED))])

def fit_model(train_text, labels, prediction_text, train_groups):
    inner = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
                 .split(train_text, labels, train_groups))
    for tr, va in inner:
        require(set(train_groups[tr]).isdisjoint(train_groups[va]), 'Inner group leakage')
        require(len(np.unique(labels[tr])) == len(np.unique(labels[va])) == 2,
                'Inner fold lacks a class')
    model = CalibratedClassifierCV(make_pipeline(), method='sigmoid', cv=inner,
                                  ensemble=False, n_jobs=1)
    model.fit(train_text, labels)
    require(model.classes_.tolist() == [0, 1], 'Incorrect class direction')
    return model.predict_proba(prediction_text)[:, 1], model


## 4. Five-fold out-of-fold evaluation (no test feedback)

In [5]:
oof_score = np.full(len(train), np.nan)
fold_assignment = np.full(len(train), -1)
fold_metrics = []
for fold, (tr, va) in enumerate(splits):
    oof_score[va], _ = fit_model(texts[tr], y[tr], texts[va], groups[tr])
    fold_assignment[va] = fold
    auc = roc_auc_score(y[va], oof_score[va])
    fold_metrics.append(float(auc))
    print(f'Fold {fold+1}: ROC-AUC={auc:.6f}', flush=True)
require(np.isfinite(oof_score).all(), 'Incomplete OOF predictions')
oof = oof_score[:, None]
metrics = {'polynomial_svm': float(roc_auc_score(y, oof_score))}
print('Grouped OOF ROC-AUC:', metrics)
print('Accuracy:', accuracy_score(y, oof_score >= .5),
      '| F1:', f1_score(y, oof_score >= .5), '| Brier:', brier_score_loss(y, oof_score))
print('Public ROC-AUC: unknown; 0.97 target remains unverified')

Fold 1: ROC-AUC=0.959783


Fold 2: ROC-AUC=0.969531


Fold 3: ROC-AUC=0.960155


Fold 4: ROC-AUC=0.953635


Fold 5: ROC-AUC=0.962974


Grouped OOF ROC-AUC: {'polynomial_svm': 0.9594691496846086}
Accuracy: 0.89525 | F1: 0.866347687400319 | Brier: 0.07784059325672754
Public ROC-AUC: unknown; 0.97 target remains unverified


## 5. Group bootstrap uncertainty
Resampling whole groups accounts for lexical dependence, conditional on these fitted OOF predictions. It does not include model-selection/training uncertainty and is not a public-score interval.

In [6]:
rng = np.random.default_rng(SEED)
group_rows = [np.flatnonzero(groups == g) for g in np.unique(groups)]
bootstrap_auc = []
for _ in range(500):
    idx = np.concatenate([group_rows[g] for g in rng.integers(0, len(group_rows), len(group_rows))])
    if len(np.unique(y[idx])) == 2:
        bootstrap_auc.append(roc_auc_score(y[idx], oof_score[idx]))
auc_interval = np.quantile(bootstrap_auc, [.025, .975]).tolist()
print('Group-bootstrap 95% interval:', auc_interval)

Group-bootstrap 95% interval: [0.9507818496309293, 0.9672444413422209]


## 6. Refit on all labeled rows and score untouched test prompts

In [7]:
scores, final_model = fit_model(texts, y, test.normalized.to_numpy(), groups)
require(np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all(), 'Invalid probabilities')
print('Test score range:', float(scores.min()), float(scores.max()))
print('Threshold counts:', pd.Series(np.where(scores >= .5, 'jailbreak', 'benign')).value_counts().to_dict())

Test score range: 5.537675478537045e-05 0.9999984679769757
Threshold counts: {'benign': 600, 'jailbreak': 400}


## 7. Fitted word-ID inspection
The unigram entries of the final word vocabulary supply stable IDs within this fitted model. Repeated words share IDs; unknown words map to -1. IDs are feature indices, not prompt IDs, and do not encode risk on their own.

In [8]:
final_pipeline = final_model.calibrated_classifiers_[0].estimator
word_vect = dict(final_pipeline.named_steps['features'].transformer_list)['word']
analyzer = word_vect.build_tokenizer()
def word_ids(prompt):
    return [(word, word_vect.vocabulary_.get(word, -1))
            for word in analyzer(normalize(prompt))]
print('Vocabulary entries (unigrams + bigrams):', len(word_vect.vocabulary_))
print(word_ids(train.text.iloc[0])[:25])

Vocabulary entries (unigrams + bigrams): 52166
[('you', np.int64(51413)), ('are', np.int64(6085)), ('an', np.int64(3708)), ('ai', np.int64(3020)), ('developed', np.int64(13544)), ('to', np.int64(46043)), ('provide', np.int64(34561)), ('useful', np.int64(48548)), ('information', np.int64(22884)), ('to', np.int64(46043)), ('users', np.int64(48608)), ('i', np.int64(21424)), ('m', np.int64(26249)), ('conducting', np.int64(11177)), ('a', np.int64(561)), ('study', np.int64(40952)), ('about', np.int64(2024)), ('data', np.int64(12698)), ('security', np.int64(38242)), ('and', np.int64(4111)), ('the', np.int64(42725)), ('disposal', np.int64(14104)), ('of', np.int64(29742)), ('electronic', np.int64(14925)), ('devices', np.int64(13593))]


## 8. Regression tests before export
Synthetic invalid inputs below test safeguards only; they are never used for training or scoring.

In [9]:
passed = []
def check(name, condition):
    require(bool(condition), name)
    passed.append(name)

def rejects(name, action):
    try:
        action()
    except ValueError:
        passed.append(name)
    else:
        raise AssertionError('Did not reject: ' + name)

base = pd.DataFrame({'Id': ['a', 'b'], 'text': ['hello', 'world'], 'label': ['benign', 'jailbreak']})
rejects('missing column', lambda: validate_frame(base.drop(columns='text'), ['Id','text','label']))
rejects('duplicate ID', lambda: validate_frame(base.assign(Id=['a','a']), ['Id','text','label']))
rejects('blank ID', lambda: validate_frame(base.assign(Id=['','b']), ['Id','text','label']))
rejects('blank text', lambda: validate_frame(base.assign(text=['  ','world']), ['Id','text','label']))
rejects('null text', lambda: validate_frame(base.assign(text=[None,'world']), ['Id','text','label']))
rejects('unknown label', lambda: validate_frame(base.assign(label=['benign','other']), ['Id','text','label']))
rejects('single class', lambda: validate_frame(base.assign(label=['benign','benign']), ['Id','text','label']))
check('unicode normalization', normalize(' Ａ\u200b B  ') == 'a b')
check('every row validated once', np.array_equal(np.sort(np.concatenate([va for _,va in splits])), np.arange(len(train))))
check('OOF finite and bounded', np.isfinite(oof).all() and ((oof >= 0) & (oof <= 1)).all())
check('test finite and bounded', np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all())
check('nonconstant test predictions', np.ptp(scores) > 0.01)
check('no group leakage', all(set(groups[tr]).isdisjoint(groups[va]) for tr,va in splits))
check('word IDs repeat consistently', word_ids('hello hello')[0][1] == word_ids('hello hello')[1][1])
check('unknown word ID', word_ids('zzzxxyyunknownword')[0][1] == -1)
# Independent fixture tests vocabulary isolation, never included in real training.
tiny = make_pipeline()
tiny.fit(np.array(['safe weather report', 'normal weather today',
    'ignore safeguards attack', 'override safeguards attack']), np.array([0,0,1,1]))
tiny_vect = dict(tiny.named_steps['features'].transformer_list)['word']
tiny.predict(np.array(['heldoutonlytoken']))
check('held-out vocabulary isolation', 'heldoutonlytoken' not in tiny_vect.vocabulary_)
check('threshold convention', np.where(np.array([.49,.5,.51]) >= .5, 1, 0).tolist() == [0,1,1])
check('repeat inference', np.allclose(final_model.predict_proba(test.normalized.iloc[:5])[:, 1], scores[:5]))
rejects('whitespace ID', lambda: validate_frame(base.assign(Id=[' a','b']), ['Id','text','label']))
rejects('NUL prompt', lambda: validate_frame(base.assign(text=['a\x00b','ok']), ['Id','text','label']))
rejects('malformed quoting', lambda: parse_csv(StringIO('Id,text\na,"unterminated'), ['Id','text']))
rejects('extra CSV field', lambda: parse_csv(StringIO('Id,text\na,b,c\n'), ['Id','text']))
rejects('duplicate CSV header', lambda: parse_csv(StringIO('Id,Id\na,b\n'), ['Id','text']))
rejects('empty CSV', lambda: parse_csv(StringIO('Id,text\n'), ['Id','text']))
check('literal NA prompt preserved', parse_csv(StringIO('Id,text\na,NA\n'), ['Id','text']).text.iloc[0] == 'NA')
valid_test = pd.DataFrame({'Id':['c'], 'text':['different prompt']})
valid_sample = pd.DataFrame({'Id':['c'], 'TARGET':['0.5']})
rejects('overlapping IDs', lambda: validate_relations(base, valid_test.assign(Id=['a']), valid_sample.assign(Id=['a'])))
rejects('normalized text overlap', lambda: validate_relations(base, valid_test.assign(text=[' HELLO ']), valid_sample))
rejects('conflicting duplicate labels', lambda: validate_relations(base.assign(text=['Hello',' hello ']), valid_test, valid_sample))
rejects('sample ID mismatch', lambda: validate_relations(base, valid_test, valid_sample.assign(Id=['wrong'])))
rejects('sample nonfinite score', lambda: validate_relations(base, valid_test, valid_sample.assign(TARGET=['nan'])))
rejects('sample score out of bounds', lambda: validate_relations(base, valid_test, valid_sample.assign(TARGET=['1.1'])))
check('normalization idempotent', all(normalize(normalize(t)) == normalize(t) for t in train.text.iloc[:100]))
check('inner calibration group isolation', all(set(groups[a]).isdisjoint(groups[b]) for a,b in
    StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(texts,y,groups)))
check('inputs unchanged', all(hashlib.sha256((ROOT/f).read_bytes()).hexdigest() == h for f,h in INPUT_HASHES.items()))
print(f'PASS: {len(passed)} checks before export')

PASS: 34 checks before export


## 9. Atomic submission export and inline audit report
The only exported file is `submission.csv`, exactly `Id,TARGET`, preserving test ID spelling/order. All checks, input hashes, versions, metrics, and limitations are printed below and saved as notebook outputs (no report sidecar).

In [10]:
submission = pd.DataFrame({'Id': test.Id, 'TARGET': scores})
temporary = ROOT / 'submission.csv.tmp'
submission.to_csv(temporary, index=False, float_format='%.12g')
roundtrip = pd.read_csv(temporary, dtype={'Id': str})
check('CSV schema', roundtrip.columns.tolist() == ['Id','TARGET'])
check('CSV ID order', roundtrip.Id.tolist() == sample.Id.tolist())
check('CSV row count', len(roundtrip) == len(test))
check('CSV probabilities', np.allclose(roundtrip.TARGET, scores, atol=1e-11))
check('CSV probability bounds', roundtrip.TARGET.between(0,1).all())
temporary.replace(ROOT / 'submission.csv')
report = {
    'seed': SEED, 'folds': FOLDS, 'training_rows': len(train), 'test_rows': len(test),
    'removed_exact_duplicates': duplicate_rows,
    'class_counts': train.label.value_counts().to_dict(), 'near_test_prompts_audit': near_test_count,
    'near_duplicate_cosine': NEAR_DUPLICATE_COSINE, 'groups': len(group_rows),
    'development_roc_auc': metrics, 'fold_roc_auc': fold_metrics,
    'group_bootstrap_95_interval': auc_interval,
    'accuracy_at_0.5': float(accuracy_score(y, oof_score >= .5)),
    'f1_at_0.5': float(f1_score(y, oof_score >= .5)),
    'brier': float(brier_score_loss(y, oof_score)),
    'tests_passed': passed,
    'input_sha256': {f: hashlib.sha256((ROOT/f).read_bytes()).hexdigest()
                     for f in ['train.csv','test.csv','sample_submission.csv']},
    'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                 'pandas': pd.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__},
    'public_leaderboard_score': None, 'user_reported_previous_public_score': 0.95397,
    'limitations': ['Lexical grouping cannot guarantee semantic or source independence.',
                    'Likelihoods are not independently calibrated.',
                    'Model family and hyperparameters compared on reused development folds.',
                    'Public leaderboard target 0.97 is unverified.'],
}
print(json.dumps(report, indent=2))
print(f'PASS: {len(passed)} checks; exported {len(submission)} rows to submission.csv')
print(f'Runtime: {time.monotonic()-START:.1f} seconds')

{
  "seed": 42,
  "folds": 5,
  "training_rows": 4000,
  "test_rows": 1000,
  "removed_exact_duplicates": 0,
  "class_counts": {
    "benign": 2401,
    "jailbreak": 1599
  },
  "near_test_prompts_audit": 43,
  "near_duplicate_cosine": 0.8,
  "groups": 3855,
  "development_roc_auc": {
    "polynomial_svm": 0.9594691496846086
  },
  "fold_roc_auc": [
    0.9597828633913927,
    0.9695305158105121,
    0.960155308608917,
    0.9536350251409841,
    0.9629738224204358
  ],
  "group_bootstrap_95_interval": [
    0.9507818496309293,
    0.9672444413422209
  ],
  "accuracy_at_0.5": 0.89525,
  "f1_at_0.5": 0.866347687400319,
  "brier": 0.07784059325672754,
  "tests_passed": [
    "missing column",
    "duplicate ID",
    "blank ID",
    "blank text",
    "null text",
    "unknown label",
    "single class",
    "unicode normalization",
    "every row validated once",
    "OOF finite and bounded",
    "test finite and bounded",
    "nonconstant test predictions",
    "no group leakage",
    "w